# populariteit bepalen van tedtalk videos

# eerst de data ophalen en opschonen

In [180]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
import joblib
from _datetime import datetime
import re

df = pd.read_csv("Kaggle_TED_video_metadata_balanced.csv")
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   title          600 non-null    object
 1   tags           590 non-null    object
 2   views          600 non-null    int64 
 3   likes          600 non-null    int64 
 4   dislikes       600 non-null    int64 
 5   comment_count  600 non-null    int64 
 6   published_at   600 non-null    object
 7   duration       600 non-null    object
 8   category_id    600 non-null    int64 
dtypes: int64(5), object(4)
memory usage: 42.3+ KB


,views,likes,dislikes,comment_count,category_id
count,6.000000e+02,6.000000e+02,600.0,600.000000,600.000000
mean,1.066151e+06,2.263473e+04,0.0,1070.870000,25.030000
std,3.611008e+06,1.000825e+05,0.0,4055.503954,4.812244
min,1.000000e+00,0.000000e+00,0.0,0.000000,1.000000
25%,5.922150e+04,6.242500e+02,0.0,86.000000,22.000000
50%,1.332595e+05,1.846000e+03,0.0,202.000000,27.000000
75%,4.101668e+05,6.382250e+03,0.0,511.750000,28.000000
max,5.562224e+07,1.921445e+06,0.0,77980.000000,29.000000


##### met de column title kan het model niet veel. Daarom maak ik er een column van met de hoeveelheid karakters in de titel.

In [181]:
df["title_length"] = df['title'].apply(len)
df.head(5)

,title,tags,views,likes,dislikes,comment_count,published_at,duration,category_id,title_length
0,Stories from a home for terminally ill childre...,"TED Talk,TED Talks,Children,Community,Death,Fa...",77455,1768,0,49,2017-03-24T15:32:48Z,PT15M19S,29,60
1,Why our screens make us less happy | Adam Alter,"TEDTalk,TEDTalks,Addiction,Computers,Interface...",800326,20579,0,572,2017-08-01T15:29:04Z,PT9M30S,22,47
2,A tribute to nurses | Carolyn Jones,"TEDTalk,TEDTalks,Cancer,Community,Compassion,D...",87635,1877,0,48,2017-05-30T18:17:56Z,PT10M49S,22,35
3,"Asking for help is a strength, not a weakness ...","TEDTalk,TEDTalks,Children,Communication,Commun...",190840,4726,0,187,2017-04-12T15:17:51Z,PT11M56S,22,67
4,Don't feel sorry for refugees -- believe in th...,"TEDTalk,TEDTalks,Children,Global issues,Humani...",98523,2669,0,226,2017-07-25T15:06:25Z,PT14M14S,29,62


##### de title is nutteloos voor machine learning, dus die halen we weg. Ook zijn de dislikes allemaal 0, omdat youtube dit uitgeschakeld heeft. Daarom verwijder ik deze column ook.

In [182]:
df = df.drop(columns=["title", "dislikes"], axis=1)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   tags           590 non-null    object
 1   views          600 non-null    int64 
 2   likes          600 non-null    int64 
 3   comment_count  600 non-null    int64 
 4   published_at   600 non-null    object
 5   duration       600 non-null    object
 6   category_id    600 non-null    int64 
 7   title_length   600 non-null    int64 
dtypes: int64(5), object(3)
memory usage: 37.6+ KB


##### de column "tags" bevat 10 null waarden. Hier ga ik echter niks aan doen, omdat dat natuurlijk voorkomt in de dataset. Er worden in de realiteit video's geopload zonder tags. Verder zijn er geen null waarden.

##### de views, likes en comment_count moeten gescaled worden voor optimalisatie

In [183]:
scaler = StandardScaler()
df[["views", "likes", "comment_count"]] = scaler.fit_transform(df[["views", "likes", "comment_count"]])
df.head(5)

,tags,views,likes,comment_count,published_at,duration,category_id,title_length
0,"TED Talk,TED Talks,Children,Community,Death,Fa...",-0.274029,-0.208669,-0.252181,2017-03-24T15:32:48Z,PT15M19S,29,60
1,"TEDTalk,TEDTalks,Addiction,Computers,Interface...",-0.073677,-0.020557,-0.123113,2017-08-01T15:29:04Z,PT9M30S,22,47
2,"TEDTalk,TEDTalks,Cancer,Community,Compassion,D...",-0.271208,-0.207579,-0.252428,2017-05-30T18:17:56Z,PT10M49S,22,35
3,"TEDTalk,TEDTalks,Children,Communication,Commun...",-0.242603,-0.179089,-0.218125,2017-04-12T15:17:51Z,PT11M56S,22,67
4,"TEDTalk,TEDTalks,Children,Global issues,Humani...",-0.268190,-0.199659,-0.208501,2017-07-25T15:06:25Z,PT14M14S,29,62


##### De tag column is nog niet leesbaar voor machine learning. Als ik one-hot encoding zou toepassen, heb ik 591 comlumns. Dit is niet handig omdat ik dan weer andere tags heb voor mijn eigen dataset. Daarom maak ik 1 column met de hoeveelheid tags.

In [184]:
df['num_tags'] = df['tags'].apply(lambda x: len(x.split(',')) if isinstance(x, str) else 0)
df = df.drop("tags", axis=1)
df.head(5)

,views,likes,comment_count,published_at,duration,category_id,title_length,num_tags
0,-0.274029,-0.208669,-0.252181,2017-03-24T15:32:48Z,PT15M19S,29,60,18
1,-0.073677,-0.020557,-0.123113,2017-08-01T15:29:04Z,PT9M30S,22,47,9
2,-0.271208,-0.207579,-0.252428,2017-05-30T18:17:56Z,PT10M49S,22,35,15
3,-0.242603,-0.179089,-0.218125,2017-04-12T15:17:51Z,PT11M56S,22,67,12
4,-0.268190,-0.199659,-0.208501,2017-07-25T15:06:25Z,PT14M14S,29,62,9


##### De published_at column is niet bruikbaar voor machine learning, daarom pas ik er wat feature engineering op toe. Eerst de dag, maand, jaar en uur scheiden.

In [185]:
df['published_at'] = pd.to_datetime(df['published_at'])

df['year_published'] = df['published_at'].dt.year
df['month_published'] = df['published_at'].dt.month
df['day_published'] = df['published_at'].dt.day
df['hour_published'] = df['published_at'].dt.hour

##### Ook kan ik de dag van de week opslaan, en de tijd sinds upload in een aparte column stoppen. de column published at hebben we niet meer nodig.

In [186]:
df['published_at'] = df['published_at'].dt.tz_localize(None)
df['days_since_published'] = (datetime.now() - df['published_at']).dt.days

df = df.drop("published_at", axis= 1)
df.head(5)

,views,likes,comment_count,duration,category_id,title_length,num_tags,year_published,month_published,day_published,hour_published,days_since_published
0,-0.274029,-0.208669,-0.252181,PT15M19S,29,60,18,2017,3,24,15,2747
1,-0.073677,-0.020557,-0.123113,PT9M30S,22,47,9,2017,8,1,15,2617
2,-0.271208,-0.207579,-0.252428,PT10M49S,22,35,15,2017,5,30,18,2680
3,-0.242603,-0.179089,-0.218125,PT11M56S,22,67,12,2017,4,12,15,2728
4,-0.268190,-0.199659,-0.208501,PT14M14S,29,62,9,2017,7,25,15,2624


#### met de duration kan ik niet veel in deze format. Daarom ga ik het omzetten naar een leesbaar format

In [187]:
df['duration_seconds'] = df['duration'].apply(lambda x: (int(re.match(r'PT(\d+)M', x).group(1)) * 60 if isinstance(x, str) and re.match(r'PT(\d+)M', x) else 0) + 
                                                            (int(re.match(r'PT(\d+)S', x).group(1)) if isinstance(x, str) and re.match(r'PT(\d+)S', x) else 0))
print(df[['duration', 'duration_seconds']])

     duration  duration_seconds
0    PT15M19S               900
1     PT9M30S               540
2    PT10M49S               600
3    PT11M56S               660
4    PT14M14S               840
..        ...               ...
595  PT24M18S              1440
596  PT20M30S              1200
597  PT10M34S               600
598   PT6M19S               360
599  PT20M20S              1200

[600 rows x 2 columns]


##### De nieuwe waardes schalen: